# 2026-11-02 信号：幅值换算＋两频变式＋泄漏（重建版）

> 原笔记本因 11/2 晚助手目录整理误删；本文件按 11/2 讲义 §3 代码重建，代码与已跑通流程一致。
> 完成的实验：幅值换算 2/N（含 DC/Nyquist 例外）、两频 T=1s 变式、2.0 vs 2.3Hz 泄漏对比、超声连接一句。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
# 3.1 幅值换算：amp = |X|·2/N；DC(k=0) 与 Nyquist(k=N/2) 例外不乘 2
fs, T = 100, 1.0
t = np.arange(int(fs*T)) / fs
x = 0.5 + np.sin(2*np.pi*2*t)          # 直流 0.5 + 2Hz 幅值 1
X = np.fft.rfft(x)
f = np.fft.rfftfreq(len(x), d=1/fs)
N = len(x)
print("0Hz :", f[0], "|X|/N =", np.abs(X[0])/N)          # 预期 ≈ 0.5
i = int(np.argmax(np.abs(X)[1:])) + 1
print("峰  :", f[i], "2|X|/N =", 2*np.abs(X[i])/N)        # 预期 2.0 Hz、≈1.0

plt.plot(f, np.abs(X))
plt.xlabel("freq (Hz)"); plt.ylabel("|X|"); plt.title("0.5 + sin(2pi·2t) 频谱")
plt.show()


In [ ]:
# 3.3 两频变式：T 从 2s 改成 1s（N=100，分辨率 1Hz）
fs, T = 100, 1.0
t = np.arange(0, T, 1/fs)
s = np.sin(2*np.pi*2*t) + 0.5*np.sin(2*np.pi*5*t)
S = np.fft.rfft(s)
f2 = np.fft.rfftfreq(len(s), d=1/fs)
amp = np.abs(S) * 2 / len(s)
for i in np.argsort(amp)[-4:]:
    print(f"f={f2[i]:.1f}Hz amp={amp[i]:.2f}")

plt.plot(f2, amp)
plt.xlabel("freq (Hz)"); plt.ylabel("amplitude"); plt.title("2Hz + 0.5·5Hz, T=1s")
plt.show()


In [ ]:
# 3.4 泄漏：2.0 vs 2.3Hz（fs=100, T=1s, N=100）
for f0 in (2.0, 2.3):
    t = np.arange(0, 1.0, 1/100)
    x = np.sin(2*np.pi*f0*t)
    X = np.fft.rfft(x)
    fr = np.fft.rfftfreq(len(x), d=1/100)
    a = np.abs(X) * 2 / len(x)
    top = np.argsort(a)[-4:][::-1]
    print(f"f0={f0}Hz →", [(round(fr[i],1), round(a[i],2)) for i in top])
    plt.plot(fr, a, marker="o", label=f"f0={f0}Hz")

plt.xlabel("freq (Hz)"); plt.ylabel("amplitude"); plt.legend()
plt.title("泄漏对比：2.0Hz vs 2.3Hz")
plt.show()


## 3.5 与超声的连接（一句话）

`RF 原始信号 → 波束形成(DAS) → 包络 → 对数压缩 → B 模式图`。
FFT 是"看信号里有什么频率成分"（发射频率、谐波、噪声落在哪一段）；滤波是"把不想要的频段去掉"。两者都不是直接出图的那一步，但决定了图干不干净。
